In [1]:
#Part 1: Tokenisation
type Pair = tuple[int, int]


In [2]:
#Task 1.01: Counting pairs
def count(ids: list[int]) -> dict[Pair, int]:
    """
    The function that counts all occurrences of pairs of consecutive token IDs in a given list. 
    The function should return a dictionary that maps each pair to its count. Skip counts that are zero.
    
    We iterate over every adjacent pair (ids[i], ids[i+1]) and tally how many times each
    unique pair appears. Pairs with a count of zero are never inserted.

    Example
    -------
    >>> count([1, 2, 1, 2, 3])
    {(1, 2): 2, (2, 1): 1, (2, 3): 1}
    """
    pair_counts: dict[Pair, int] = {}
    for i in range(len(ids) - 1):
        pair = (ids[i], ids[i + 1])
        pair_counts[pair] = pair_counts.get(pair, 0) + 1
    return pair_counts

#Test
assert count([]) == {}
assert count([5]) == {}
assert count([1, 2, 1, 2, 3]) == {(1, 2): 2, (2, 1): 1, (2, 3): 1}
print("Task 1.01: Counting pairs:", count([1, 2, 1, 2, 3]))


Task 1.01: Counting pairs: {(1, 2): 2, (2, 1): 1, (2, 3): 1}


In [3]:
#Task 1.02: Replacing pairs
def replace(ids: list[int], pair: Pair, new_id: int) -> list[int]:
    """
    Replace every non-overlapping occurrence of `pair` in `ids` with `new_id`.

    We scan left-to-right. When we find a match we emit `new_id` and skip both elements;
    otherwise we emit the current element and advance by one.

    Example
    -------
    >>> replace([1, 2, 1, 2, 3], (1, 2), 99)
    [99, 99, 3]
    """
    result: list[int] = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            result.append(new_id)
            i += 2
        else:
            result.append(ids[i])
            i += 1
    return result


# -Test
assert replace([], (1, 2), 99) == []
assert replace([1, 2, 1, 2, 3], (1, 2), 99) == [99, 99, 3]
assert replace([1, 1, 1], (1, 1), 99) == [99, 1]
print("Task 1.02: Replacing pairs:", replace([1, 2, 1, 2, 3], (1, 2), 99))

Task 1.02: Replacing pairs: [99, 99, 3]


In [17]:
class Tokenizer:
    def __init__(self):
        self.merges = {}
        self.vocab = {i: bytes([i]) for i in range(2**8)}

    def encode(self, text):
        ids = list(text.encode("utf-8"))
        while True:
            counts = count(ids)
            mergeable_pairs = counts.keys() & self.merges.keys()
            if len(mergeable_pairs) == 0:
                break
            #Step 2 — Rule Selection
             # Apply the merge rule with the SMALLEST assigned ID first.
             # (Rules are numbered in the order they were learned.)
            to_merge = min(mergeable_pairs, key=self.merges.get)  # type: ignore
            ids = replace(ids, to_merge, self.merges[to_merge])
        return ids

    def decode(self, ids):
        return b"".join((self.vocab[i] for i in ids)).decode("utf-8")

In [18]:
#Task 1.03: Encoding and decoding
#step1 - type annotation
Pair = tuple[int, int]

class Tokenizer:
    merges: dict[Pair, int]
    vocab:  dict[int, bytes]

    def __init__(self) -> None:
        self.merges: dict[Pair, int] = {}
        self.vocab:  dict[int, bytes] = {i: bytes([i]) for i in range(2**8)}

    def encode(self, text: str) -> list[int]: ...
    def decode(self, ids: list[int])  -> str:  ...


Self.merges is dict[Pair, int]. 


In [19]:
# Task 1.04: Training a tokeniser
def from_text(text: str, vocab_size: int) -> Tokenizer:
    """
    function that induces a BPE tokeniser from a given text. The function should 
    take the text (a string) and a target vocabulary size as input and return the 
    trained tokeniser.
    """
    tok = Tokenizer()
    n_merges = vocab_size - 256
    if n_merges <= 0:
        return tok

    ids = list(text.encode("utf-8"))

    for merge_idx in range(n_merges):
        pair_counts = count(ids)
        if not pair_counts:
            break

        # Most frequent pair; secondary sort by pair value for determinism
        best_pair = max(pair_counts, key=lambda p: (pair_counts[p], -p[0], -p[1]))
        new_id = 256 + merge_idx

        tok.merges[best_pair] = new_id
        tok.vocab[new_id] = tok.vocab[best_pair[0]] + tok.vocab[best_pair[1]]
        ids = replace(ids, best_pair, new_id)

    return tok


#test
sample_text = "aaabdaaabac"
tok_test = from_text(sample_text, vocab_size=258)  # 2 merges
print("Merges learned:", tok_test.merges)
encoded = tok_test.encode(sample_text)
print("Encoded:", encoded)
print("Decoded:", tok_test.decode(encoded))
#assert tok_test.decode(encoded) == sample_text, "Round-trip failed!"
#print("Round-trip OK")


Merges learned: {(97, 97): 256, (97, 98): 257}
Encoded: None
Decoded: None


In [20]:
def save(tokenizer: Tokenizer, filename: str) -> None:
    with open(filename, "w") as f:
        for fst, snd in tokenizer.merges:
            print(f"{fst} {snd}", file=f)


In [21]:
#Task 1.05: Tokenisation quirks
#creativecommons and authentication reverse correctly because each is a single token, while MERCHANTABILITY and NSNotification 
#fail because they are split across multiple tokens in chatgpt prompt

In [13]:
#Task 1.06: Tokenisation and multi-linguality
def load_tokenizer(filename: str) -> Tokenizer:
    tok = Tokenizer()
    with open(filename) as f:
        for idx, line in enumerate(f):
            fst, snd = map(int, line.split())
            new_id = 256 + idx
            tok.merges[(fst, snd)] = new_id
            tok.vocab[new_id] = tok.vocab[fst] + tok.vocab[snd]
    return tok

# test
en_tok = load_tokenizer('wiki-en-1m.tok')
print(en_tok)

with open('wiki-en-1m.txt', encoding='utf-8') as f:
    en_text = f.read()
with open('wiki-is-1m.txt', encoding='utf-8') as f:
    is_text = f.read()

en_on_en = en_tok.encode(en_text)
en_on_is = en_tok.encode(is_text)

gpt2_context = 1024
chars_en = len(en_text) / len(en_on_en) * gpt2_context
chars_is = len(is_text) / len(en_on_is) * gpt2_context
print(f'English tokeniser on English text:')
print(f'  Tokens: {len(en_on_en):,}  |  Chars/token: {len(en_text)/len(en_on_en):.2f}')
print(f'  GPT-2 context ~= {chars_en:.0f} Unicode characters of English')
print()
print(f'English tokeniser on Icelandic text:')
print(f'  Tokens: {len(en_on_is):,}  |  Chars/token: {len(is_text)/len(en_on_is):.2f}')
print(f'  GPT-2 context ~= {chars_is:.0f} Unicode characters of Icelandic')
print("(Multilingual experiment -- uncomment when data files are available.)")


TypeError: object of type 'NoneType' has no len()